# Laser-loop frequency response — A6

Measure the combined piezo/EOM loop while it is locked. The measurement includes the DAC/output electronics, both controllers and actuators, laser, ADC, internal mixer/FIR, CORDIC and **the active discriminator stage**. It is not a separate measurement of the FALC alone.

Load the matching A6 package using `laser_pll_bringup.ipynb`, reacquire lock and retain its `ol` object. Run these cells individually. Loading a bitstream interrupts the lock; importing this helper does not. Leave controller filters, gains, offsets and FPGA stage fixed throughout each sweep. Stage 2 includes its frequency-discriminator term; stage 3/4 measures its active unwrapped phase mapping.


In [ ]:
import importlib
from pathlib import Path
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import laser_pll
import laser_pll_loop_sweep as fra

laser_pll = importlib.reload(laser_pll)
fra = importlib.reload(fra)
print("Helper:", fra.__file__)
print("FPGA:", hex(ol.pll_acquisition_status.read(8) >> 24))  # 0xa6
print(laser_pll.acquisition_status(ol))
print("Output directory:", Path.cwd())


## Signals and sign convention

The injected sine is `d`. The original discriminator output is `e`; the DAC command is `u=e+d`. Hardware demodulates **e and u simultaneously**, in the same signed 16-bit DAC-code units and with the same reference. A saved DC baseline is removed only from the monitor probes. The main feedback path has no new pipeline register.

For negative feedback, `E=-L*U`. Therefore `L=-E/U` is the return ratio around the complete loop, `S=U/(U-E)=1/(1+L)` is the sensitivity, and `T=-E/(U-E)=L/(1+L)`. I and Q are sine/cosine lock-in sums; `I+1j*Q` gives positive phase for a leading response. Do not conjugate it.

The old eight-lane generator advances eight samples per RF clock. Lane 0 is used here, so its DDS step is 0.457764 Hz. Integration uses nearly whole cycles at the **actual** frequency, with less than half a fabric clock of rounding. Each interval fits up to 17.48 s. A 1.067 ms envelope ramp precedes settling and follows integration.


In [ ]:
# First test one point, at the current locked operating point.
# 32 command codes is about 0.977 mV peak for a nominal 2 Vpp DAC span.
# At the /160 slope this corresponds to ~0.49 rad of detector-equivalent phase.
point = fra.measure_point(ol, 1e3, amplitude_codes=32)
e, u = point["error"], point["drive"]
print("Actual frequency (Hz):", point["frequency_hz"])
print("E, U (peak DAC codes):", e, u)
print("L = -E/U:", -e/u if abs(u) else np.nan)


Check the scope during the first point: the laser should remain locked and both error/controller outputs should stay away from clipping. Hardware rejects clipped, invalid-phase, stalled-DAC or dropped-probe records. These checks are not an optical lock detector. Repeating at half the injection amplitude tests whether the measured response is linear; a discrepancy can also mean inadequate signal-to-noise.

The default low-frequency settling is five cycles and integration is sixteen cycles; both have a 20 ms minimum. At 20 Hz, one point takes about 1.05 s before repeats. Increase the settling cycles if the loop has a slow transient, and increase integration/repeats if results scatter. The fine discriminator remains at the full 245.76 MHz clock, without a measurement low-pass filter in the loop.


In [ ]:
# A small amplitude check before committing to a full sweep.
half_point = fra.measure_point(ol, 1e3, amplitude_codes=16)
print("L at 32 codes:", -point["error"]/point["drive"])
print("L at 16 codes:", -half_point["error"]/half_point["drive"])


In [ ]:
# Record the physical settings here so later curves can be compared.
filter_settings = "FALC: enter switches, gain, input offset; piezo settings; FPGA stage/shift"
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = Path(f"laser_loop_{stamp}.csv")
notes_path = csv_path.with_suffix(".txt")
notes_path.write_text(filter_settings + "\n" + repr(laser_pll.acquisition_status(ol)) + "\n")
frequencies_hz = np.geomspace(20, 1e6, 61)
# For a later detailed sweep near the observed line:
# frequencies_hz = np.linspace(150e3, 200e3, 51)
rows = fra.sweep(
    ol, frequencies_hz,
    amplitude_codes=32, repeats=3,
    settle_cycles=5, measure_cycles=16,
    minimum_settle_s=0.02, minimum_measure_s=0.02,
    csv_path=csv_path,
)
print("Saved:", csv_path.resolve())


The CSV is checkpointed after each completed frequency. It stores requested/actual frequencies, E/U/injection phasors in peak DAC codes, the complex L and S estimates, integration time, FPGA gain/stage words and repeat standard errors. If interrupted, the helper ramps only the test tone to zero and keeps feedback enabled. The records already written remain available.

`reliable` means both measured phasors exceed five times their estimated standard errors across repeats. It is a statistical screening flag, not a calibration or a guarantee: deterministic distortion and common measurement errors can pass it. At low frequency, large loop gain makes U small, so -E/U can become poorly conditioned. Inspect the underlying E/U values and repeat at another amplitude or for longer.


In [ ]:
f_hz = np.array([r["frequency_hz"] for r in rows])
L = np.array([r["loop_re"] + 1j*r["loop_im"] for r in rows])
S = np.array([r["sensitivity_re"] + 1j*r["sensitivity_im"] for r in rows])
reliable = np.array([r["reliable"] for r in rows])
finite = np.isfinite(L.real) & np.isfinite(L.imag)
phase_deg = np.full(len(L), np.nan)
phase_deg[finite] = np.rad2deg(np.unwrap(np.angle(L[finite])))

fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
axes[0].semilogx(f_hz, 20*np.log10(np.maximum(abs(L), 1e-15)), ".-")
axes[0].axhline(0, color="0.5", ls="--")
axes[0].set_ylabel("Loop gain |L| (dB)")
axes[1].semilogx(f_hz, phase_deg, ".-")
axes[1].axhline(-180, color="0.5", ls="--")
axes[1].set_ylabel("Loop phase (deg)")
axes[2].semilogx(f_hz, 20*np.log10(np.maximum(abs(S), 1e-15)), ".-")
axes[2].axhline(0, color="0.5", ls="--")
axes[2].set_ylabel("Sensitivity |S| (dB)")
axes[2].set_xlabel("Actual modulation frequency (Hz)")
for ax in axes:
    ax.grid(True, which="both", alpha=0.3)
    for frequency in f_hz[~reliable]: ax.axvline(frequency, color="red", alpha=0.15)
fig.suptitle(filter_settings)
fig.tight_layout()
fig.savefig(csv_path.with_suffix(".png"), dpi=180)
plt.show()


Look for sensitivity peaking (`|S|>1`) and the phase near unity loop gain. Keep the displayed phase branch and any multiple crossings in mind before reading a phase margin. The measurement includes the discriminator delay; no digital-delay correction should be applied to L. Measure several points above a suspected crossover rather than extrapolating past 1 MHz.

After each sweep, change one analog setting and save a new file. Return to a known stable setting if peaking grows or the optical lock degrades. This measures the combined response of both actuators at their shared input; it cannot uniquely separate the piezo and EOM transfer functions.


In [ ]:
# Optional manual cleanup after an interrupted cell. This does not mute the PLL.
fra.stop(ol)


## Optional I/Q noise capture

This is a separate measurement with the test tone off. I/Q is baseband relative to the fabric DDS, before CORDIC and DAC scaling. D=7 with one million samples gives 1.92 MS/s for 0.521 s and 7.32 Hz Welch bins below. D=4 gives 15.36 MS/s for faster noise.

The October 9 captures showed substantial phase motion near 175 kHz. D=12 averaging converted that motion into apparent AM and nearly zero vectors, making slow phase unwrapping ambiguous. Use sufficiently fast, nonvanishing I/Q; unwrap phase **before** additional offline low-pass filtering/decimation. Compare D=6 and D=7 to check convergence. The boxcar has limited anti-alias rejection, and an in-loop PSD is not an independent optical-noise-floor measurement.


In [ ]:
from scipy.signal import welch
fra.stop(ol)
iq = laser_pll.capture_iq(ol, samples=1000000, decimation_log2=7)
noise_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
raw_path = Path(f"laser_iq_{noise_stamp}.npz")
np.savez(raw_path, **iq)
phase_rad = np.unwrap(np.angle(iq["i_adc_counts"] + 1j*iq["q_adc_counts"]))
f_noise_hz, s_phase = welch(phase_rad, fs=iq["sample_rate_hz"], nperseg=262144,
                          detrend="constant", scaling="density")
psd_path = Path(f"laser_phase_psd_{noise_stamp}.csv")
np.savetxt(psd_path, np.column_stack((f_noise_hz, s_phase)), delimiter=",",
           header="frequency_Hz,phase_PSD_rad2_per_Hz", comments="")
plt.loglog(f_noise_hz[1:], s_phase[1:])
plt.xlabel("Fourier frequency (Hz)"); plt.ylabel("Phase PSD (rad²/Hz)")
plt.grid(True, which="both", alpha=0.3); plt.show()
print("Saved:", raw_path.resolve(), psd_path.resolve())
